# Theory

A plain random train-test split is usually fine — but not always. If a dataset has an important categorical attribute with an uneven distribution, a purely random split can, just by chance, produce a training set and test set with meaningfully different proportions of that category. This can bias both training (the model under-learns a minority group) and evaluation (the test set no longer reflects the real population).

Stratified sampling fixes this by preserving the proportion of a chosen category (the stratum) in every split — train, validation, and test all mirror the same category distribution as the full dataset.

**When to Use It**

The target is categorical and imbalanced (e.g., 95% "no fraud" vs. 5% "fraud" — classification).

A key categorical feature is imbalanced and its distribution matters for fair evaluation (e.g., Fuel_Type, Owner_Type).

For regression targets (like our continuous Price), we can still stratify — by first binning the continuous target into ranges (e.g., price quartiles), then stratifying on those bins.

**Applying It to This Dataset**

Looking at used_cars_dataset.csv, a few columns are natural stratification candidates:

Fuel_Type — Petrol dominates, with Diesel, Hybrid, and CNG appearing far less often. A random split could easily leave the test set with zero or very few Hybrid/CNG cars, making it impossible to fairly judge how well the model prices those cars.
Owner_Type — "First Owner" is likely far more common than "Fourth & Above Owner." A skewed split could under-represent higher-owner-count cars in training.
Price (binned) — since Price is continuous, we can create price bands (e.g., budget / mid-range / premium / luxury) and stratify on those bands, ensuring the model is trained and tested fairly across the full price spectrum — not just the most common price range.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("used_cars_dataset.csv").drop(columns=["Car_ID"])

print("Fuel_Type distribution (full dataset):")
print(df["Fuel_Type"].value_counts(normalize=True))

X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df["Fuel_Type"]     # preserves Fuel_Type proportions in both splits
)

print("\nFuel_Type distribution (train):")
print(X_train["Fuel_Type"].value_counts(normalize=True))
print("\nFuel_Type distribution (test):")
print(X_test["Fuel_Type"].value_counts(normalize=True))

**Stratifying on a Binned Continuous Target (Price)**

In [ ]:
import numpy as np

# Create price bands purely for stratification purposes (not used as a model feature)
price_bins = pd.qcut(df["Price"], q=4, labels=["budget", "mid", "high", "luxury"])
print(price_bins.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=price_bins    # ensures each price band is proportionally represented in both sets
)

# Confirm balance
train_bins = pd.qcut(y_train, q=4, labels=["budget", "mid", "high", "luxury"])
test_bins = pd.qcut(y_test, q=4, labels=["budget", "mid", "high", "luxury"])
print("Train price band distribution:\n", train_bins.value_counts(normalize=True))
print("Test price band distribution:\n", test_bins.value_counts(normalize=True))